# Sprint 3 Campus — Stream and observe FieldCare

Continue the secured FieldCare service you carried through Sprint 2. This notebook supports C14–C19: when streaming helps, how to observe real incremental delivery, which operational fields to retain, what to exclude, and how to reuse Module A’s existing evaluation criteria.

Supported requests use real Chroma retrieval, LangGraph, and OpenRouter. Each admitted model request and initial embedding may use provider quota. The evaluator is the preserved deterministic Module A application path; it checks original criteria and does not pretend to judge generated prose.

**Colab runtime:** `ServiceProcess` runs Uvicorn in a managed child process bound to loopback, checks readiness with a timeout, and provides an explicit stop. Do not run `uvicorn.run()` in a cell or expose the service through ngrok; a foreground server call would prevent that cell from returning. The streaming client reads until a terminal event or its bounded timeout, then the notebook can continue to logging and evaluation.


## Campus lessons in Notion

- [When streaming helps](https://app.notion.com/p/3ea5a55972ad81fd9cdee92887860939?pvs=204)
- [Enabling streaming on the scaffold](https://app.notion.com/p/3ea5a55972ad8118a84ef55a5d48442e?pvs=204)
- [What to log and why](https://app.notion.com/p/3ea5a55972ad817f865ec69c602dd137?pvs=204)
- [What NOT to log](https://app.notion.com/p/3ea5a55972ad8168bb14fe42cd0c288f?pvs=204)
- [Closing the loop with existing evaluation](https://app.notion.com/p/3ea5a55972ad816f89b7f0bd61f6d601?pvs=204)
- [Observable streaming service checkpoint](https://app.notion.com/p/3ea5a55972ad81d9b6dafb3e76ff75fd?pvs=204)


## 1. Set up the same service

Run once in a fresh Colab runtime. Load your secured Sprint 2 checkpoint when available. Without it, the notebook labels the supplied starter as a recovery workspace; it does not claim your previous changes.

In [ ]:
import json
import os
from pathlib import Path
import runpy
import secrets
import shutil
import subprocess
import sys
from zipfile import ZipFile

REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path("/content/ai-app-dev_Mod-B")
PROJECT = Path("/content/fieldcare-sprint-3-campus")
SPRINT2_CHECKPOINT = ""  # Optional uploaded secured Sprint 2 source checkpoint.
LOG_PATH = PROJECT / "logs" / "requests.jsonl"

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
else:
    # Colab runtimes persist /content; refresh an existing clone before reinstalling it.
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))  # Make the editable package visible to this running kernel.

from module_b.openrouter import DEFAULT_CHAT_MODEL, require_openrouter_key
require_openrouter_key(prompt=True)
os.environ.setdefault("OPENROUTER_MODEL", DEFAULT_CHAT_MODEL)

if SPRINT2_CHECKPOINT:
    PROJECT.mkdir(parents=True, exist_ok=True)
    with ZipFile(SPRINT2_CHECKPOINT) as archive:
        archive.extractall(PROJECT)
elif not PROJECT.exists():
    shutil.copytree(REPO / "examples" / "fieldcare", PROJECT)

settings_path = PROJECT / "app" / "security_settings.py"
if not settings_path.exists():
    settings_path.write_text("""from module_b.security import LimitPolicy
CALLER_ENV = {"dispatch": "FIELDCARE_DISPATCH_KEY", "partner": "FIELDCARE_PARTNER_KEY"}
POLICY = LimitPolicy(allowance=2, window_seconds=60)
""")
settings = runpy.run_path(str(settings_path))
CALLERS = settings["CALLER_ENV"]
caller_keys = {label: secrets.token_urlsafe(32) for label in CALLERS}
for label, variable in CALLERS.items():
    os.environ[variable] = caller_keys[label]

print("Module B source:", REPO)
print("Sprint 3 workspace:", PROJECT)
print("Starting point:", "Sprint 2 checkpoint" if SPRINT2_CHECKPOINT else "labelled recovery workspace")

## 2. Add the streaming operation and safe observation

Open app/stream_routes.py in Colab’s Files panel. Trace the request schema, deterministic JSON clarification, LangGraph retrieval call, OpenRouter SSE iteration, and the stream's metadata, delta, usage, and complete/error events before sending a request. Metadata carries the status and citation IDs needed by a streaming UI. The registration cell adds that real route before the caller middleware inventories protected POST paths, then places the metadata logger outside the guard.

In [ ]:
import ast
from module_b.observability import FIELDS, ObservationMiddleware
from module_b.runtime import ServiceProcess
import httpx
import time

stream_source = REPO / "examples" / "patterns" / "sprint_3" / "stream_routes.py"
stream_file = PROJECT / "app" / "stream_routes.py"
if not stream_file.exists():
    shutil.copyfile(stream_source, stream_file)

main_path = PROJECT / "app" / "main.py"
main_source = main_path.read_text()
route_registration = """
from app.stream_routes import router as stream_router
app.include_router(stream_router)
"""
security_marker = "from module_b.security import SecurityMiddleware"
if "diagnose_stream" not in main_source:
    if security_marker in main_source:
        main_source = main_source.replace(security_marker, route_registration + "\n" + security_marker, 1)
    else:
        main_source += route_registration + """
from module_b.security import SecurityMiddleware, protected_post_paths
from app.security_settings import CALLER_ENV, POLICY
app.add_middleware(SecurityMiddleware, caller_env=CALLER_ENV,
                   protected_paths=protected_post_paths(app), policy=POLICY)
"""
tree = ast.parse(main_source)
imports_os = any(
    isinstance(node, ast.Import) and any(alias.name == "os" for alias in node.names)
    or isinstance(node, ast.ImportFrom) and node.module == "os"
    for node in ast.walk(tree)
)
if not imports_os:
    lines = main_source.splitlines(keepends=True)
    docstring = ast.get_docstring(tree)
    insertion_line = tree.body[0].end_lineno if docstring else 0
    lines.insert(insertion_line, "import os\n")
    main_source = "".join(lines)
observation_marker = "# Module B request observation middleware"
if observation_marker not in main_source:
    main_source += f"""
{observation_marker}
from module_b.observability import ObservationMiddleware
app.add_middleware(
    ObservationMiddleware,
    path={str(LOG_PATH)!r},
    routes=("/v1/diagnose", "/v1/diagnose-stream"),
    approved_models=(os.environ["OPENROUTER_MODEL"],),
)
"""
main_path.write_text(main_source)

def start_service():
    service = ServiceProcess(
        project_dir=PROJECT,
        env={variable: caller_keys[label] for label, variable in CALLERS.items()},
    )
    service.start()
    return service

service = start_service()
labels = list(CALLERS)
stream_caller = labels[0]
buffered_caller = labels[1] if len(labels) > 1 else labels[0]
payload = {
    "question": "Which filter and airflow checks are documented?",
    "equipment_id": "EQ-FC-1002",
}

## 3. Read actual incremental provider events

Run the stream request. Each printed delta is text received from the live OpenRouter response. The event count, arrival times, answer, provider usage, and terminal result depend on this run. Partial text is not a completed answer unless the final event is complete.

In [ ]:
events = []
arrival_seconds = []
started = time.perf_counter()
with httpx.Client(base_url=service.base_url, timeout=120) as client:
    with client.stream(
        "POST",
        "/v1/diagnose-stream",
        json=payload,
        headers={"X-API-Key": caller_keys[stream_caller]},
    ) as response:
        request_id = response.headers.get("x-request-id")
        print("HTTP:", response.status_code, "request:", request_id)
        for line in response.iter_lines():
            if not line:
                continue
            event = json.loads(line)
            events.append(event)
            arrival_seconds.append(time.perf_counter() - started)
            print(event)

text = "".join(event.get("text", "") for event in events if event.get("type") == "delta")
terminal = next((event for event in reversed(events)
                 if event.get("type") in {"complete", "error"}), None)
usage = next((event for event in events if event.get("type") == "usage"), None)
print("Generated text:", text)
print("Terminal event:", terminal)
print("Provider usage:", usage.get("tokens") if usage else None,
      "(unknown when absent)")

## 4. Compare with the buffered response

The next request uses a second caller when one is configured, so a one-request per-key Sprint 2 policy can still show both paths. It makes another real provider call. Compare the response contract and citations; do not expect identical wording.

In [ ]:
with httpx.Client(base_url=service.base_url, timeout=120) as client:
    response = client.post(
        "/v1/diagnose",
        json=payload,
        headers={"X-API-Key": caller_keys[buffered_caller]},
    )
buffered_result = response.json()
print("HTTP:", response.status_code)
print("Status:", buffered_result.get("status"))
print("Answer:", buffered_result.get("answer"))
print("Citations:", buffered_result.get("citations"))

## 5. Inspect the safe operational record

The observation middleware assigns a runtime request ID and stores allowlisted fields. These records contain metadata, not the prompt or provider key. Use the exact request ID to correlate the stream with its terminal record. Missing token usage remains unknown, not zero.

In [ ]:
records = [json.loads(line) for line in LOG_PATH.read_text().splitlines() if line]
for record in records:
    print({field: record.get(field) for field in FIELDS})
print("Request IDs:", [row["request_id"] for row in records])
print("Prompt retained:", any("prompt" in row for row in records))
print("Authorization header retained:", any("headers" in row for row in records))

### Check the allowlist against synthetic sensitive fields

Start from the actual streamed request’s safe record, then add a visibly synthetic prompt and header to a local copy. The computation checks the allowlist without creating a second service event or sending the marker to OpenRouter.

In [ ]:
from module_b.observability import safe_record

marker = "SYNTHETIC-NOT-A-CREDENTIAL"
observed = next(record for record in records if record["request_id"] == request_id)
candidate = {
    **observed,
    "_approved_models": (os.environ["OPENROUTER_MODEL"],),
    "prompt": "Synthetic private marker " + marker,
    "headers": {"X-API-Key": marker},
}
minimized = safe_record(
    candidate,
    routes=("/v1/diagnose", "/v1/diagnose-stream"),
)
assert minimized["request_id"] == request_id
assert minimized["model"] == observed["model"]
assert marker not in repr(minimized)
print("Based on actual request:", request_id)
print("Retained fields:", minimized)
print("Marker in retained record:", marker in repr(minimized))

## 6. Reuse the original evaluation criteria

Choose an approved Module A case and a related companion case in the next cell. Inspect the original expectations, then run the preserved deterministic evaluator on the saved design. These checks concern application evidence, tools, and decision flags; they do not grade the generated stream text.

In [ ]:
from module_b._module_a import project as original
from module_b.evaluation import load_pipeline, evaluate_selected

design_path = PROJECT / "data" / "pipeline_design.json"
design_path.parent.mkdir(parents=True, exist_ok=True)
if not design_path.exists():
    design_path.write_text(json.dumps(original.default_pipeline_design(), indent=2) + "\\n")

pipeline = load_pipeline(design_path)
SELECTED_CASE = "EVAL-FC-005"  # Choose an approved case relevant to your concern.
COMPANION_CASE = "EVAL-FC-003"  # Choose a related case that protects neighboring behavior.
if SELECTED_CASE == COMPANION_CASE:
    raise ValueError("Choose two different evaluation cases.")
case_ids = [SELECTED_CASE, COMPANION_CASE]
report = evaluate_selected(design_path, case_ids)
for case_id in case_ids:
    case = original.eval_case_by_id(pipeline.env, case_id)
    print("Original criteria:", case["eval_id"], case["expected_behavior"])
for row in report["rows"]:
    print({
        "eval_id": row["eval_id"],
        "pipeline_pass": row["pipeline_pass"],
        "missing_required_docs": row["missing_required_docs"],
        "missing_required_tools": row["missing_required_tools"],
        "missing_expected_flags": row["missing_expected_flags"],
    })
print("Design revision:", report["design_revision"])
print("Evaluation behavior is deterministic application evidence; it does not score generated prose.")

## 7. Checkpoint and carry forward

Record your stream-versus-buffered decision, one actual observation, the fields retained and excluded, and what the selected evaluation cases establish. Export the app, data, and safe logs; save the notebook separately.

In [ ]:
service.stop()
from zipfile import ZIP_DEFLATED, ZipFile

CHECKPOINT_OUT = Path("/content/sprint-3-campus-checkpoint.zip")
with ZipFile(CHECKPOINT_OUT, "w", ZIP_DEFLATED) as archive:
    for folder in ("app", "data", "logs"):
        path = PROJECT / folder
        if not path.exists():
            continue
        for item in path.rglob("*"):
            if item.is_file() and "__pycache__" not in item.parts and item.suffix != ".pyc":
                archive.write(item, item.relative_to(PROJECT))
print("Saved checkpoint:", CHECKPOINT_OUT)

### Sprint 3 checkpoint

You extended the authenticated FieldCare service with real streaming and safe operational metadata, then used the existing evaluation rules as a bounded application signal. Keep service-delivery evidence, evaluation results, and generated answer review distinct.